# TP 4 — TP noté : équations différentielles, méthodes d'Euler et de Taylor

**L3 Mathématiques · Équations différentielles et optimisation · Université d'Artois · 2026–2027**

**Durée : 1 h 45 · Barème indicatif sur 20 points**

Ce TP est **noté**. Il porte sur les TP 1 à 3 : Python, NumPy, Matplotlib, `solve_ivp` et SymPy (TP 1), méthode
d'Euler (TP 2), méthodes de Taylor (TP 3).

| Exercice | Thème | Points |
|:--|:--|:--|
| 1 | Analyse qualitative d'une équation autonome, `solve_ivp`, SymPy | 5 |
| 2 | La méthode d'Euler : erreur, borne théorique, ordre | 6 |
| 3 | Les méthodes de Taylor : dérivées totales, ordre | 6 |
| 4 | Une famille de méthodes à un pas | 3 |

L'exercice 1 est indépendant des autres. Les exercices 2, 3 et 4 portent sur le **même** problème de Cauchy
(sauf la question 3.a).
Un **aide-mémoire** des commandes Python utiles se trouve à la fin du notebook (partie 5).

**Légende.** 📌 rappel de cours · 📖 cellule fournie, à exécuter · ✏️ à compléter (code ou réponse rédigée)

✏️ **Nom :** … &nbsp;&nbsp;&nbsp; **Prénom :** …

## Consignes

1. **Commencez par exécuter la cellule de mise en route** (partie 0), puis exécutez les cellules **dans l'ordre**
   (Maj + Entrée).
2. Complétez les cellules ✏️ : remplacez les `...` par votre code, et écrivez vos réponses rédigées dans les
   cellules « Votre réponse » (double-clic pour écrire, Maj + Entrée pour valider). Des réponses courtes suffisent.
3. **Gardez les noms** des fonctions et des variables demandées : ils servent à la correction. Vous pouvez ajouter
   des cellules si besoin.
4. **Documents autorisés :** vos notebooks des TP 1 à 3 et l'aide-mémoire de la partie 5. Les assistants d'IA, les
   messageries et les autres sites ne sont pas autorisés.
5. **Sauvegardez souvent.** Le notebook est enregistré *dans le navigateur de cet ordinateur* : téléchargez-le
   (**File → Download**) régulièrement, au moins toutes les 30 minutes.
6. **Pour rendre votre travail :**
   - **Kernel → Restart Kernel and Run All Cells**, et vérifiez que tout s'exécute. L'exécution s'arrête à la
     première erreur : si une cellule inachevée en produit une, mettez ses lignes en commentaire (sélectionnez-les,
     puis Ctrl + /). Le code commenté reste lisible pour la correction ;
   - **File → Download**, renommez le fichier `TP4_NOM_Prenom.ipynb`, puis envoyez-le par e-mail à
     santiago.torooquendo@univ-artois.fr (objet : « TP4 NOM Prénom »).

## 0. Mise en route 📖

Exécutez cette cellule : elle importe les modules et définit la fonction `champ_directions` du TP 1.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import sympy as sp


def champ_directions(f, t_min, t_max, y_min, y_max, n=20, ax=None):
    """Trace le champ de directions de y' = f(t, y) dans le repère ax (fonction du TP 1) et renvoie ax."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    T, Y = np.meshgrid(np.linspace(t_min, t_max, n), np.linspace(y_min, y_max, n))
    P = f(T, Y)
    norme = np.sqrt(1 + P**2)
    ax.quiver(T, Y, 1 / norme, P / norme, angles="xy", color="0.6",
              headwidth=0, headlength=0, headaxislength=0, pivot="mid")
    ax.set_xlim(t_min, t_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("$t$")
    ax.set_ylabel("$y$")
    return ax


print("✔ Mise en route terminée : vous pouvez commencer.")

## Exercice 1 — Analyse qualitative d'une équation autonome (5 points)

On considère l'équation différentielle autonome

$$y' = -(y + 1)(y + 3).$$

> 📌 **Rappel (cours et TP 1).** On considère une équation autonome $y' = f(y)$.
> - Un **point d'équilibre** est un zéro $a$ de $f$ : la fonction constante $y \equiv a$ est alors solution.
> - Là où $f > 0$, les solutions sont croissantes ; là où $f < 0$, elles sont décroissantes.
> - **Définition.** Le point d'équilibre $a$ est **stable** lorsque toute donnée initiale suffisamment proche de $a$
>   engendre une solution globale, et que
>   $$\forall \varepsilon > 0,\ \exists \delta > 0 :\quad \big(y \text{ solution},\ |y(0) - a| \leq \delta\big)
>   \ \Longrightarrow\ \sup_{t > 0} |y(t) - a| \leq \varepsilon.$$
>   Autrement dit : partir près de $a$ garantit de rester près de $a$, uniformément en temps. Sinon, $a$ est
>   **instable**.
>
> **Théorème (admis).** Soit $f$ de classe $\mathcal{C}^1$ et $a$ un point d'équilibre, c'est-à-dire $f(a) = 0$.
> - Si $f'(a) < 0$, le point d'équilibre $a$ est stable.
> - Si $f'(a) > 0$, le point d'équilibre $a$ est instable.
> - Si $f'(a) = 0$, on ne peut pas conclure : il faut une étude plus fine de $f$.
>
> *Idée* : près de $a$, $f(y) \approx f'(a)\,(y - a)$, donc l'écart $z = y - a$ vérifie à peu près $z' = f'(a)\,z$,
> dont les solutions $z(0)\,e^{f'(a)\,t}$ se rapprochent de $0$ si $f'(a) < 0$ et s'en éloignent si $f'(a) > 0$.
>
> En Python, le second membre s'écrit toujours comme une fonction de **deux** arguments `(t, y)`, dans cet ordre,
> même quand il ne dépend pas de $t$.

---
#### ✏️ 1.a) Points d'équilibre (1,5 point)

Écrire le second membre `f_auto(t, y)`, qui doit fonctionner quand `y` est un tableau NumPy. Déterminer les points
d'équilibre, puis ranger les équilibres stables dans la liste `stables` et les équilibres instables dans la liste
`instables` (vous pouvez utiliser le théorème admis du rappel).

In [ ]:
def f_auto(t, y):
    ...


equilibres = ...     # les points d'équilibre
stables = ...        # liste des équilibres stables
instables = ...      # liste des équilibres instables
print(equilibres, stables, instables)

---
#### ✏️ 1.b) Champ de directions et solutions (1,5 point)

Sur une même figure, dans la fenêtre $[0, 2] \times [-4, 0]$, tracer :

- le champ de directions de l'équation, avec la fonction `champ_directions` fournie ;
- les solutions calculées avec `solve_ivp` sur $[0, 2]$ pour
  $y(0) \in \{-3{,}5 ;\ -3 ;\ -2{,}5 ;\ -2 ;\ -1{,}5 ;\ -1 ;\ -0{,}5 ;\ 0\}$ ;
- les droites d'équilibre.

Ajouter une légende.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
champ_directions(f_auto, 0, 2, -4, 0, ax=ax)
t_eval = np.linspace(0, 2, 201)
for y0 in [-3.5, -3, -2.5, -2, -1.5, -1, -0.5, 0]:
    ...          # résoudre avec solve_ivp, puis tracer la solution
...              # droites d'équilibre et légende
plt.show()

**Question.** Décrire le comportement des solutions selon la valeur de $y(0)$. La figure est-elle cohérente avec la
question 1.a, au sens de la **définition** de la stabilité ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

---
#### ✏️ 1.c) Vérification avec SymPy (1 point)

On admet que la solution du problème de Cauchy $y' = -(y + 1)(y + 3)$, $y(0) = -2$, est

$$y(t) = -3 + \frac{2}{1 + e^{-2t}}.$$

Avec SymPy :

1. vérifier que cette fonction est bien la solution : calculer `residu`, l'expression simplifiée de
   $y'(t) + \big(y(t) + 1\big)\big(y(t) + 3\big)$, et `valeur_0`, la valeur de $y(0)$ ;
2. calculer `limite`, la limite de $y(t)$ quand $t \to +\infty$.

In [ ]:
ts = sp.symbols("t")         # « s » pour symbolique : on n'écrase pas les tableaux t
y_expr = ...                 # l'expression de y(t)
residu = ...
valeur_0 = ...
limite = ...
print(residu, valeur_0, limite)

**Question.** La valeur de `limite` est-elle cohérente avec la question 1.a ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

---
#### ✏️ 1.d) Une solution qui explose (1 point)

Pour $y(0) = -3{,}5$, `solve_ivp` n'atteint pas $t = 2$. Résoudre à nouveau ce problème sur $[0, 2]$ (sans `t_eval`),
afficher `sol.status` et `sol.message`, et ranger dans `t_arret` le dernier instant atteint par le solveur.

On admet que, pour $y(0) = y_0 < -3$, la solution tend vers $-\infty$ quand $t$ tend vers
$T^* = \dfrac{1}{2} \ln\left(\dfrac{y_0 + 1}{y_0 + 3}\right)$. Calculer `T_etoile` pour $y_0 = -3{,}5$, et comparer.

In [ ]:
sol = ...
print(sol.status, sol.message)
t_arret = ...
T_etoile = ...
print(t_arret, T_etoile)

**Question.** Le solveur est-il en défaut ? Sur quel intervalle cette solution est-elle définie ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## Exercice 2 — La méthode d'Euler (6 points)

Dans les exercices 2, 3 et 4, on considère le problème de Cauchy

$$y' = \frac{2}{t}\, y + t^2 e^t, \quad t \in [1, 2], \qquad y(1) = 0,$$

dont la solution exacte est $y(t) = t^2\,(e^t - e)$. Exécutez la cellule suivante 📖 :

In [ ]:
def y_exacte(t):
    """Solution exacte du problème des exercices 2 à 4."""
    return t**2 * (np.exp(t) - np.e)

> 📌 **Rappel (TP 2) — la méthode d'Euler.** On découpe $[t_0, T]$ en $N$ pas de longueur $h = \dfrac{T - t_0}{N}$ :
> $t_i = t_0 + i\,h$ pour $i = 0, \dots, N$. La méthode d'Euler calcule des valeurs approchées $w_i \approx y(t_i)$ par
> $$w_0 = y_0, \qquad w_{i+1} = w_i + h\, f(t_i, w_i), \quad i = 0, 1, \dots, N - 1.$$

---
#### ✏️ 2.a) Programmer la méthode (1,5 point)

Écrire le second membre `f(t, y)` du problème, puis la fonction `euler(f, t0, T, y0, N)`, qui renvoie les tableaux
`t` $= (t_0, \dots, t_N)$ et `w` $= (w_0, \dots, w_N)$, de taille $N + 1$, pour **n'importe quel** second membre `f`.

In [ ]:
def f(t, y):
    ...


def euler(f, t0, T, y0, N):
    """Méthode d'Euler pour y' = f(t, y), y(t0) = y0, sur [t0, T] avec N pas.
    Renvoie les tableaux t et w, de taille N + 1."""
    ...
    return t, w

---
#### ✏️ 2.b) Erreur (1 point)

Avec $h = 0{,}1$, calculer la solution d'Euler (`t`, `w`), puis, sans boucle, le tableau `erreurs` des
$|y(t_i) - w_i|$ et l'erreur maximale `E_euler`.

In [ ]:
t, w = ...
erreurs = ...
E_euler = ...
print("E_euler =", E_euler)

---
#### ✏️ 2.c) La borne d'erreur du cours (1,5 point)

> 📌 **Rappel (TP 2) — borne d'erreur.** Si $f$ est continue et $L$-lipschitzienne par rapport à $y$ sur
> $[t_0, T] \times \mathbb{R}$, et si $|y''(t)| \leq M$ pour tout $t \in [t_0, T]$, alors, pour tout $i$,
> $$|y(t_i) - w_i| \leq \frac{hM}{2L}\left(e^{L(t_i - t_0)} - 1\right).$$
> Si $\partial f / \partial y$ est continue et vérifie $|\partial f / \partial y| \leq L$ sur
> $[t_0, T] \times \mathbb{R}$, alors $f$ est $L$-lipschitzienne par rapport à $y$.

1. Déterminer une constante de Lipschitz `L` de $f$ par rapport à $y$ (justifier dans la cellule de réponse).
2. Calculer $y''(t)$ (à la main ou avec SymPy), puis `M` $= \max_{t \in [1, 2]} |y''(t)|$ (numériquement, sur une
   grille fine de $[1, 2]$).
3. En déduire `h_borne` : le plus grand pas $h$ pour lequel la borne **garantit** $|y(t_i) - w_i| \leq 0{,}1$ pour
   tout $i$.

In [ ]:
L = ...
M = ...
h_borne = ...
print(L, M, h_borne)

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

---
#### ✏️ 2.d) Ordre de convergence (2 points)

> 📌 **Rappel (TP 2) — ordre.** Si l'erreur maximale vérifie $E(h) \approx C\,h^p$, la méthode est **d'ordre $p$** :
> en échelle log-log, les points $(h, E(h))$ sont alignés sur une droite de pente $p$, et quand on divise $h$ par 2,
> $$p \approx \log_2\left(\frac{E(h)}{E(h/2)}\right).$$

1. Pour chaque $N$ de `N_liste`, calculer l'erreur maximale $E(h) = \max_i |y(t_i) - w_i|$ de la méthode d'Euler :
   tableau `E_liste`, et tableau `h_liste` des pas correspondants.
2. Calculer, sans boucle, le tableau `ordres` des $\log_2\big(E(h)/E(h/2)\big)$.
3. Tracer $E$ en fonction de $h$ en échelle log-log.
4. Avec le modèle $E(h) \approx C\,h$ et la dernière valeur de `E_liste`, estimer la constante `C`, puis le nombre de
   pas `N_estime` nécessaire pour avoir $E(h) \leq 0{,}1$. Comparer avec le nombre de pas $1/h_{\text{borne}}$ donné
   par la borne de la question 2.c.

In [ ]:
N_liste = [10, 20, 40, 80, 160, 320]
h_liste = ...
E_liste = ...
ordres = ...
print(ordres)

fig, ax = plt.subplots(figsize=(7, 5))
...                  # E_liste en fonction de h_liste, en échelle log-log
ax.set_xlabel("pas $h$")
ax.set_ylabel("erreur maximale $E(h)$")
ax.grid(True, which="both", alpha=0.3)
plt.show()

C = ...
N_estime = ...
print(C, N_estime, 1 / h_borne)

**Question.** Quel est l'ordre observé ? Que pensez-vous de la borne de la question 2.c ?

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## Exercice 3 — Les méthodes de Taylor (6 points)

> 📌 **Rappel (TP 3) — méthode de Taylor d'ordre $n$.** On pose $f^{(0)} = f$ et on définit les **dérivées totales**
> $$f^{(k)} = \partial_t f^{(k-1)} + f\, \partial_y f^{(k-1)}, \qquad \text{de sorte que} \qquad
> y^{(k+1)}(t) = f^{(k)}\big(t, y(t)\big).$$
> La méthode de Taylor d'ordre $n$ est la méthode à un pas
> $$w_{i+1} = w_i + h\, T^{(n)}(t_i, w_i), \qquad
> T^{(n)}(t, w) = \sum_{k=0}^{n-1} \frac{h^k}{(k+1)!}\, f^{(k)}(t, w)
> = f(t, w) + \frac{h}{2}\, f'(t, w) + \dots + \frac{h^{n-1}}{n!}\, f^{(n-1)}(t, w).$$
> Si $y$ est de classe $\mathcal{C}^{n+1}$, elle est d'ordre $n$ ; pour $n = 1$, c'est la méthode d'Euler.

---
#### ✏️ 3.a) Un pas « à la main » (1 point)

On considère le problème $x' = g(t, x) = -t\,x^2$, $x(0) = 1$, de solution exacte $x(t) = \dfrac{2}{2 + t^2}$.

1. Calculer **à la main** la dérivée totale $g'(t, x)$, puis compléter les fonctions `g` et `g1` ($g$ et $g'$).
2. Calculer `x1`, la valeur approchée de $x(0{,}1)$ obtenue avec **un** pas de la méthode de Taylor d'ordre 2
   ($h = 0{,}1$), et `erreur_x1`, l'erreur commise.

In [ ]:
def g(t, x):
    ...


def g1(t, x):
    """Dérivée totale g'(t, x)."""
    ...


h = 0.1
x1 = ...
erreur_x1 = ...
print(x1, erreur_x1)

---
#### ✏️ 3.b) Les dérivées totales avec SymPy (1,5 point)

On revient au problème $y' = f(t, y) = \dfrac{2}{t}\,y + t^2 e^t$. Avec SymPy, construire la liste `derivees_expr`
$= [f, f', f'', f''']$ des **expressions** des dérivées totales, puis la liste `derivees` des **fonctions** Python
de $(t, y)$ correspondantes (avec `sp.lambdify`).

In [ ]:
ts, ys = sp.symbols("t y")
f_expr = ...
derivees_expr = [f_expr]
...                          # ajouter f', f'' et f''' à la liste
derivees = ...               # liste des 4 fonctions Python de (t, y)

for expr in derivees_expr:
    display(sp.simplify(expr))

---
#### ✏️ 3.c) Programmer la méthode de Taylor (2 points)

1. Écrire la fonction `taylor(derivees, t0, T, y0, N)` : la méthode de Taylor d'ordre $n$ = `len(derivees)`, où
   `derivees` $= [f, f', \dots, f^{(n-1)}]$ est une liste de fonctions de $(t, y)$. Mêmes sorties que `euler`.
2. Avec $h = 0{,}1$, calculer les solutions de Taylor d'ordre 2 (`w_T2`) et d'ordre 4 (`w_T4`), puis leurs erreurs
   maximales `E_T2` et `E_T4`. Comparer avec `E_euler`.

In [ ]:
def taylor(derivees, t0, T, y0, N):
    """Méthode de Taylor d'ordre n = len(derivees) pour y' = f(t, y), y(t0) = y0, sur [t0, T] avec N pas.
    derivees = [f, f', ..., f^(n-1)]. Renvoie les tableaux t et w, de taille N + 1."""
    ...
    return t, w


t, w_T2 = ...
t, w_T4 = ...
E_T2 = ...
E_T4 = ...
print(E_euler, E_T2, E_T4)

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

---
#### ✏️ 3.d) Ordre des méthodes de Taylor (1,5 point)

Comme à la question 2.d, calculer les erreurs maximales des méthodes de Taylor d'ordre 2 et d'ordre 4 pour
$N \in \{10, 20, 40, 80, 160\}$, puis les tableaux `ordres_T2` et `ordres_T4` des ordres observés.
Sont-ils conformes à la théorie ?

In [ ]:
N_liste2 = [10, 20, 40, 80, 160]
E_T2_liste = ...
E_T4_liste = ...
ordres_T2 = ...
ordres_T4 = ...
print(ordres_T2)
print(ordres_T4)

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## Exercice 4 — Une famille de méthodes à un pas (3 points)

On étudie la méthode à un pas $w_{i+1} = w_i + h\,\Phi(t_i, w_i, h)$, de fonction d'incrément

$$\Phi(t, w, h) = \alpha\, f(t, w) + \beta\, f\Big(t + \frac{h}{2},\ w + \frac{h}{2}\, f(t, w)\Big)
+ \gamma\, f\big(t + h,\ w + h\, f(t, w)\big),$$

où $\alpha$, $\beta$ et $\gamma$ sont des réels de $[0, 1]$. On l'applique au problème de l'exercice 2.

> 📌 **Rappel (TP 2 et 3).** La méthode d'**Euler modifiée** (ou de Heun) du TP 2 s'écrit
> $w_{i+1} = w_i + \frac{h}{2}\big[f(t_i, w_i) + f\big(t_{i+1},\ w_i + h\, f(t_i, w_i)\big)\big]$, et la méthode du
> **point milieu** $w_{i+1} = w_i + h\, f\big(t_i + \frac{h}{2},\ w_i + \frac{h}{2}\, f(t_i, w_i)\big)$.
> On estime l'ordre d'une méthode comme à la question 2.d.

---
#### ✏️ 4.a) Trois méthodes connues (0,5 point)

Quels triplets $(\alpha, \beta, \gamma)$ donnent la méthode d'Euler, la méthode du point milieu et la méthode
d'Euler modifiée ?

In [ ]:
abc_euler = (..., ..., ...)
abc_milieu = (..., ..., ...)
abc_heun = (..., ..., ...)

---
#### ✏️ 4.b) Programmer la méthode (1 point)

Écrire la fonction `methode_abc(f, t0, T, y0, N, alpha, beta, gamma)` (mêmes sorties que `euler`), puis vérifier
qu'avec le triplet `abc_euler`, elle redonne les valeurs de la fonction `euler`.

In [ ]:
def methode_abc(f, t0, T, y0, N, alpha, beta, gamma):
    """Méthode à un pas de fonction d'incrément Phi (ci-dessus). Renvoie les tableaux t et w, de taille N + 1."""
    ...
    return t, w


t1, w1 = methode_abc(f, 1, 2, 0, 10, *abc_euler)       # *abc_euler : les trois nombres du triplet
t2, w2 = euler(f, 1, 2, 0, 10)
print(np.max(np.abs(w1 - w2)))

---
#### ✏️ 4.c) Expériences numériques (1,5 point)

Pour chacun des triplets de la liste `triplets`, calculer l'ordre observé
$p \approx \log_2\big(E(h)/E(h/2)\big)$ entre $N = 160$ et $N = 320$ pas, et le ranger dans la liste `ordres_abc`.

In [ ]:
triplets = [(1, 0, 0), (0.5, 0, 0.5), (1/3, 1/3, 1/3), (0.5, 0.5, 0), (0.2, 0.2, 0.2)]
ordres_abc = []
for alpha, beta, gamma in triplets:
    ...

for (alpha, beta, gamma), p in zip(triplets, ordres_abc):
    print(f"alpha = {alpha:.3f}, beta = {beta:.3f}, gamma = {gamma:.3f} :  ordre observé {p:.3f}")

**Question.** Au vu de ces résultats, conjecturer une condition sur $(\alpha, \beta, \gamma)$ pour que la méthode
converge, puis une condition pour qu'elle soit d'ordre 2. ★ *Bonus* : justifier ces conditions à l'aide d'un
développement de $\Phi(t, y, h)$ au voisinage de $h = 0$.

✏️ **Votre réponse :** *(double-cliquez ici pour écrire)*

## 5. Aide-mémoire

### 5.1 Python

| Commande | Explication | Exemple |
|:--|:--|:--|
| `def nom(a, b):` puis `return ...` | définit une fonction ; le bloc est indenté (4 espaces), `return` renvoie le résultat | `def carre(x):`<br>`    return x**2` |
| `lambda a, b: ...` | fonction courte, sans nom | `F = lambda t, y: t * y` |
| `x**p` | puissance $x^p$ (**jamais** `x ^ p`) | `t**2` |
| `for i in range(N):` | répète le bloc pour $i = 0, 1, \dots, N - 1$ ($N$ valeurs : $N$ est exclu) | `for i in range(3):` → 0, 1, 2 |
| `for a, b in zip(A, B):` | parcourt deux listes en parallèle | `for t_i, w_i in zip(t, w):` |
| `L = []`, `L.append(v)` | liste vide ; ajoute `v` à la fin de la liste | |
| `L[0]`, `L[-1]`, `len(L)` | premier élément (les indices commencent à 0), dernier élément, longueur | |
| `[expr for v in L]` | construit la liste des `expr` pour `v` parcourant `L` | `[k**2 for k in range(4)]` → `[0, 1, 4, 9]` |
| `a, b, c = triplet` | range les trois éléments d'un triplet dans trois variables | `a, b, c = (1, 0, 0)` |
| `fonction(*triplet)` | passe les éléments du triplet comme arguments séparés | `max(*(3, 7, 5))` → 7 |
| `abs(x)`, `int(x)` | valeur absolue ; partie entière d'un réel positif | `int(5.7)` → 5 |
| `math.factorial(k)`, `math.ceil(x)` | $k!$ ; plus petit entier $\geq x$ | `math.ceil(4.2)` → 5 |
| `math.exp(x)`, `math.log(x)` | exponentielle, logarithme népérien d'**un nombre** (pas d'un tableau) | |
| `print(f"{x:.4f}")`, `f"{x:.2e}"` | affiche `x` avec 4 décimales ; en notation scientifique | `f"{0.000123:.2e}"` → `'1.23e-04'` |

### 5.2 NumPy (`import numpy as np`)

| Commande | Explication | Exemple |
|:--|:--|:--|
| `np.array([1, 2, 3])` | tableau ; `+ - * / **` agissent terme à terme | `2 * np.array([1, 2])` → `[2, 4]` |
| `np.linspace(a, b, n)` | `n` points régulièrement espacés de `a` à `b` (inclus). $N$ pas $\Rightarrow$ `N + 1` points | `np.linspace(1, 2, 11)` : pas de 0,1 |
| `np.zeros(n)` | tableau de `n` zéros, à remplir ensuite | `w = np.zeros(11)` |
| `np.exp`, `np.log`, `np.sin`, `np.sqrt`, `np.abs` | fonctions usuelles, terme à terme sur un tableau | `np.exp(t)` |
| `np.e`, `np.pi` | les nombres $e$ et $\pi$ | |
| `np.max(x)`, `np.min(x)` | plus grand, plus petit élément | `np.max(np.abs(x - y))` : écart maximal |
| `x[i]`, `x[-1]`, `x[1:4]` | élément d'indice `i` (à partir de 0) ; dernier ; indices 1, 2, 3 (la fin est exclue) | |
| `x[:-1]`, `x[1:]` | tout sauf le dernier ; tout sauf le premier | `E[:-1] / E[1:]` : rapports de valeurs successives |
| `x[x > 0]` | masque : les éléments qui vérifient la condition | `t[w <= 0][0]` : premier $t_i$ tel que $w_i \leq 0$ |
| `np.log2(x)` | logarithme en base 2, terme à terme | `np.log2(8)` → 3.0 |
| `np.roots([a, b, c])` | racines du polynôme $a y^2 + b y + c$ (dans un ordre quelconque) | `np.roots([1, 0, -1])` → `[-1, 1]` |
| `np.sort(x)` | trie par ordre croissant | |
| `np.meshgrid(x, y)` | grille 2D, pour les champs de directions | |

### 5.3 Matplotlib (`import matplotlib.pyplot as plt`)

| Commande | Explication |
|:--|:--|
| `fig, ax = plt.subplots(figsize=(8, 5))` | crée une figure et un repère `ax` |
| `ax.plot(x, y, "o-", label="...")` | trace une courbe ; styles : `"-"` trait, `"--"` tirets, `":"` pointillés, `"o"` points, `"o-"` points reliés |
| `ax.loglog(x, y, "o-")` | même chose en échelle log-log (une loi $C h^p$ devient une droite de pente $p$) |
| `ax.semilogy(x, y)` | échelle logarithmique en ordonnée seulement |
| `ax.axhline(c, ls="--", color="crimson", label="...")` | droite horizontale $y = c$ |
| `ax.set_xlabel("$t$")`, `ax.set_ylabel(...)`, `ax.set_title(...)` | noms des axes, titre (formules entre `$...$`) |
| `ax.set_xlim(a, b)`, `ax.set_ylim(c, d)` | fenêtre d'affichage |
| `ax.legend()`, `ax.grid(True, which="both", alpha=0.3)` | légende (elle utilise les `label`), quadrillage |
| `plt.show()` | affiche la figure |
| `champ_directions(f, t_min, t_max, y_min, y_max, n=20, ax=ax)` | (fournie, partie 0) champ de directions de $y' = f(t, y)$ dans le repère `ax` |

### 5.4 SciPy : `solve_ivp` (`from scipy.integrate import solve_ivp`)

```python
sol = solve_ivp(f, (t0, T), [y0], t_eval=t_eval, rtol=1e-8, atol=1e-10)
```

| Élément | Explication |
|:--|:--|
| `f` | le second membre, fonction de `(t, y)` dans cet ordre |
| `(t0, T)` | l'intervalle de temps |
| `[y0]` | la condition initiale, **toujours dans une liste** (même pour une seule équation) |
| `t_eval` (facultatif) | les instants où l'on veut la solution |
| `rtol`, `atol` (facultatifs) | les tolérances d'erreur (par défaut `1e-3` et `1e-6`) |
| `sol.t`, `sol.y[0]` | les instants et les valeurs de la solution (`sol.y` a une ligne par inconnue) |
| `sol.status`, `sol.message` | `0` : succès ; `-1` : échec, expliqué par le message |

### 5.5 SymPy (`import sympy as sp`)

| Commande | Explication | Exemple |
|:--|:--|:--|
| `ts, ys = sp.symbols("t y")` | crée des symboles mathématiques (nommés `ts`, `ys` pour ne pas écraser les tableaux `t`, `y`) | |
| `sp.exp`, `sp.sin`, `sp.sqrt`, `sp.log`, `sp.E`, `sp.pi`, `sp.oo` | fonctions et constantes **symboliques** ($e$, $\pi$, $+\infty$) | `expr = ts**2 * sp.exp(ts)` |
| `sp.diff(expr, ts)` | dérivée (partielle) par rapport à `ts` | `sp.diff(ts * ys**2, ys)` → `2*t*y` |
| `sp.diff(expr, ts, 2)` | dérivée seconde par rapport à `ts` | |
| `sp.simplify(expr)`, `sp.expand(expr)` | simplifie ; développe | |
| `expr.subs(ts, 0)` | remplace `ts` par 0 | |
| `sp.limit(expr, ts, sp.oo)` | limite quand $t \to +\infty$ | |
| `sp.lambdify((ts, ys), expr, "numpy")` | transforme l'expression en fonction Python de `(t, y)`, utilisable avec des tableaux | `F = sp.lambdify((ts, ys), ts * ys, "numpy")` puis `F(2.0, 3.0)` → 6.0 |
| `sp.lambdify(ts, expr, "numpy")` | idem pour une expression d'une seule variable | |
| `display(expr)`, `float(expr)` | affiche l'expression en notation mathématique ; valeur décimale | |

### 5.6 Pièges fréquents

- **`NameError`** : une cellule précédente n'a pas été exécutée. En cas de doute :
  **Kernel → Restart Kernel and Run All Cells**.
- **`TypeError` avec `ellipsis`** : il reste des `...` à compléter.
- `x ^ 2` n'est **pas** une puissance en Python : écrire `x**2`.
- `math.exp(t)` ne fonctionne pas si `t` est un tableau : utiliser `np.exp(t)`. Dans une expression SymPy, utiliser
  `sp.exp` ; après `sp.lambdify`, on obtient une fonction NumPy.
- `range(N)` s'arrête à `N - 1`. Pour $N$ pas, il y a `N + 1` points : `np.linspace(t0, T, N + 1)`, `np.zeros(N + 1)`.
- `solve_ivp` : condition initiale dans une liste `[y0]` ; la solution est dans `sol.y[0]`.
- Ne pas réutiliser un nom déjà pris : un tableau appelé `f` écrase la fonction `f`, un symbole `t` écrase le tableau `t`.

---

**C'est fini !** Avant de rendre : **Kernel → Restart Kernel and Run All Cells**, puis **File → Download**, renommez
le fichier `TP4_NOM_Prenom.ipynb`, et envoyez-le par e-mail à santiago.torooquendo@univ-artois.fr
(objet : « TP4 NOM Prénom »).